<a href="https://colab.research.google.com/github/Belcamino/Ex3.Inferenza/blob/master/Inferenza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

# Verifica se CUDA (GPU NVIDIA) è disponibile e imposta il dispositivo. Se non è disponibile, usa la CPU.
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU disponibile: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU non disponibile, verrà usata la CPU.")

# La scelta del dispositivo (GPU o CPU) influenzerà le prestazioni e l'utilizzo della memoria.
# Per modelli molto piccoli e per scopi didattici, la CPU può essere sufficiente,
# ma la GPU è preferibile per modelli più grandi e inferenze più rapide.


GPU disponibile: Tesla T4


In [ ]:
# Innanzitutto, installiamo le librerie necessarie.
# `transformers` è la libreria principale per i modelli LLM.
# `sentencepiece` è spesso richiesto dai tokenizer di alcuni modelli.
!pip install transformers sentencepiece accelerate -q

# `accelerate` può aiutare a gestire l'allocazione della memoria e le prestazioni,
# specialmente su dispositivi con risorse limitate o per modelli più grandi.


### Caricamento di un Modello Instruction-Tuned con Quantizzazione a 4 bit

Per migliorare la capacità del modello di seguire le istruzioni e fornire risposte dirette, utilizzeremo un modello 'instruction-tuned'. Per mantenerlo leggero in termini di memoria, applicheremo la **quantizzazione a 4 bit**.

Useremo **Mistral-7B-Instruct-v0.2**, un modello noto per le sue buone prestazioni nel seguire le istruzioni, anche se è più grande di GPT-2.

In [ ]:
# Installiamo la libreria `bitsandbytes` che è necessaria per la quantizzazione a 4 bit.
!pip install bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.8 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Il nuovo modello instruction-tuned che useremo
model_name_new = "mistralai/Mistral-7B-Instruct-v0.2"

print(f"Scaricando e caricando il modello: {model_name_new} con quantizzazione a 4 bit")

# Definiamo la configurazione per la quantizzazione a 4 bit.
# Questo ridurrà drasticamente l'uso della memoria del modello.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Abilita il caricamento a 4 bit
    bnb_4bit_quant_type="nf4", # Tipo di quantizzazione (NF4 è raccomandato)
    bnb_4bit_compute_dtype=torch.float16, # Tipo di dato per i calcoli
    bnb_4bit_use_double_quant=True, # Usa la doppia quantizzazione per maggiore precisione
)

# Carichiamo il tokenizer per il nuovo modello.
tokenizer_new = AutoTokenizer.from_pretrained(model_name_new)

# Impostiamo il pad_token per il tokenizer se non è definito (comune per Mistral/Llama)
# e lo colleghiamo al pad_token_id del modello.
if tokenizer_new.pad_token is None:
    tokenizer_new.pad_token = tokenizer_new.eos_token

# Carichiamo il modello con la configurazione di quantizzazione.
# Usiamo `device_map="auto"` per lasciare a accelerate la gestione dell'allocazione su GPU/CPU.
model_new = AutoModelForCausalLM.from_pretrained(
    model_name_new,
    quantization_config=bnb_config,
    device_map="auto"
)

# Impostiamo il modello in modalità di valutazione.
model_new.eval()

print("Modello e tokenizer per Mistral caricati con successo!")

# Nota: il calcolo della memoria con `get_memory_footprint` potrebbe non essere accurato
# per i modelli quantizzati, ma ci dà un'indicazione.
# In pratica, un modello da 7B quantizzato a 4 bit occupa circa 3.5-4 GB di VRAM.
print(f"Memoria RAM (VRAM) usata dal modello Mistral (approssimativamente): {model_new.get_memory_footprint() / (1024**3):.2f} GB")

Scaricando e caricando il modello: mistralai/Mistral-7B-Instruct-v0.2 con quantizzazione a 4 bit


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Modello e tokenizer per Mistral caricati con successo!
Memoria RAM (VRAM) usata dal modello Mistral (approssimativamente): 3.74 GB


### Interrogazione del Modello Instruction-Tuned (Mistral)

Ora proveremo a interrogare il modello Mistral con la stessa domanda. I modelli instruction-tuned spesso beneficiano di un formato di 'chat' specifico nel prompt, che li aiuta a capire meglio il ruolo dell'utente e la richiesta.

In [ ]:
# Definiamo il prompt utilizzando il formato di chat per Mistral.
# Questo formato include ruoli (user, assistant) che aiutano il modello a capire il contesto.
messages = [
    {"role": "user", "content": "Qual è il contrario di 'grande'?"}
]

# Applichiamo il template di chat al nostro messaggio per ottenere l'input formattato.
formatted_prompt = tokenizer_new.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(f"Prompt formattato per Mistral:\n{formatted_prompt}")

# Convertiamo il prompt formattato in token numerici.
input_ids_new = tokenizer_new.encode(formatted_prompt, return_tensors="pt").to(model_new.device)

# Generiamo il testo con il nuovo modello.
with torch.no_grad():
    output_new = model_new.generate(
        input_ids_new,
        max_new_tokens=100,
        num_return_sequences=1,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        pad_token_id=tokenizer_new.pad_token_id # Usiamo il pad token del nuovo tokenizer
    )

# Decodifichiamo l'output, ignorando il prompt iniziale per mostrare solo la risposta del modello.
generated_text_new = tokenizer_new.decode(output_new[0][len(input_ids_new[0]):], skip_special_tokens=True)

print("\nTesto generato da Mistral:")
print(generated_text_new)

# Confronta questa risposta con quella di GPT-2. Dovresti notare un miglioramento significativo
# nella capacità del modello di comprendere e rispondere alla tua istruzione.

Prompt formattato per Mistral:
<s> [INST] Qual è il contrario di 'grande'? [/INST]

Testo generato da Mistral:
The opposite of "grande" which means "big" or "large" in Italian is "piccolo" which means "small" or "little".
